In [1]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    pipeline
)
from sklearn.metrics import accuracy_score
from sklearn.utils import resample
from accelerate import notebook_launcher
from datasets import load_dataset
import pandas as pd
import torch

/Users/veikka/Desktop/Master's Thesis/thesis-work/notebooks/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Finnish FinBERT sentiment model (TurkuNLP)
MODEL_NAME = "TurkuNLP/finnish-modernbert-large"

# Load tokenizer + model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

# Set device: MPS if available (Apple Silicon), else CPU
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at TurkuNLP/finnish-modernbert-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
# Create pipeline
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer,
    device=device
)

Device set to use mps


In [4]:
model_name = "TurkuNLP/finnish-modernbert-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)

cleaned_sent_dataset = pd.read_pickle('../data/finnsentiment_cleaned.pkl')

# Replace with your dataset loading
ds = cleaned_sent_dataset

In [5]:
ds.head(10)

,label,text
0,2,- Tervetuloa skotlantiin...
1,1,"...... No, oikein sopiva sattumaha se vaan oli..."
2,1,40.
3,2,Kyseessä voi olla loppuelämäsi nainen.
4,2,Sinne vaan ocean clubiin iskemään!
5,2,Itsekin pidän Keskustan kampanjointia ihan hyv...
6,1,"Kamppi, Kontula, Kluuvi"
7,1,huomista päivää odotellen..
8,0,en haluaisi että kissani vuotaa.. =)
9,0,Nyt olisi lääkitys paikallaan.


In [6]:
ds["label"].value_counts()

label
1    20287
2     1875
0     1710
Name: count, dtype: int64

In [7]:
df_0 = ds[ds['label'] == 0]
df_1 = ds[ds['label'] == 1]
df_2 = ds[ds['label'] == 2]

# # Undersample class 1
# df_1_balanced = df_1.sample(n=min(len(df_0), len(df_2)), random_state=42)

# # Combine for balanced dataset
# balanced_ds = pd.concat([df_0, df_1_balanced, df_2])

In [8]:
df_0_upsampled = resample(df_0, replace=True, n_samples=len(df_1), random_state=42)
df_2_upsampled = resample(df_2, replace=True, n_samples=len(df_1), random_state=42)

balanced_ds = pd.concat([df_1, df_0_upsampled, df_2_upsampled])

In [9]:
balanced_ds["label"].value_counts()

label
1    20287
0    20287
2    20287
Name: count, dtype: int64

In [10]:
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

# Split your DataFrame
train_df, val_df = train_test_split(balanced_ds, test_size=0.2, random_state=42)

# Convert to Hugging Face Dataset
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

# Preprocess using map (not pandas apply)
def preprocess(batch):
    enc = tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)
    enc["label"] = batch["label"]
    return enc

train_dataset = train_dataset.map(preprocess, batched=True)
val_dataset = val_dataset.map(preprocess, batched=True)

# Create DatasetDict for Trainer
ds_preprocessed = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset
})

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return {"accuracy": accuracy_score(labels, predictions)}

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    fp16=False,  # <-- set this to False
    save_strategy="epoch",
    logging_dir="./logs",
)

# Now you can use:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_preprocessed["train"],
    eval_dataset=ds_preprocessed["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

Map: 100%|██████████| 12173/12173 [00:00<00:00, 46096.08 examples/s]
/var/folders/vs/mq6j80s94v7_ts2x082jdx1w0000gn/T/ipykernel_4727/1019647292.py:42: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [11]:
# eval_results = trainer.evaluate()
# print("Evaluation results:", eval_results)

In [21]:
trainer.save_model("../data/finbert_sentiment_model")

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("../data/finbert_sentiment_model")

In [14]:
sentiment_classifier = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer,
    device="mps"
)

# Example usage:
result = sentiment_classifier("Tämä on todella hyvä uutinen!")
print(result)

Device set to use mps


[{'label': 'LABEL_0', 'score': 0.5905340313911438}]


In [15]:
print(sentiment_classifier("Kyseessä voi olla loppuelämäsi nainen."))

[{'label': 'LABEL_0', 'score': 0.5228577256202698}]


In [16]:
print(sentiment_classifier("Nyt olisi lääkitys paikallaan."))

[{'label': 'LABEL_1', 'score': 0.7063756585121155}]


In [17]:
print(sentiment_classifier("vittu mitä paskaa."))

[{'label': 'LABEL_0', 'score': 0.5409536361694336}]


In [18]:
print(sentiment_classifier("turpa kiinni."))

[{'label': 'LABEL_0', 'score': 0.8059418797492981}]


In [20]:
print(sentiment_classifier("Tämä on todella hyvä uutinen!"))

[{'label': 'LABEL_0', 'score': 0.5905340313911438}]
